# Bayesian Linear Regression

**Recitation by Dr. Stephanie Milani**

Please make a copy of this notebook in your own Google Drive. This notebook aims to revise logistic regression models using PyTorch and pandas libraries in the flower recognition application.

## Multi-class classification with logistic regression

In this exercise, we will implement a logistic regression model on multi-class classification. We will use the [Iris](https://scikit-learn.org/stable/datasets/toy_dataset.html#iris-plants-dataset) flower data set which contains different physical properties from three related species of Iris flowers collected by Edgar Anderson. It was first used by British statistician and biologist Ronald Fisher in 1936 for his pioneering study on linear discriminant analysis [1].

<div>
<img src="https://upload.wikimedia.org/wikipedia/commons/5/56/Kosaciec_szczecinkowaty_Iris_setosa.jpg" width="200"/>
<img src="https://upload.wikimedia.org/wikipedia/commons/4/41/Iris_versicolor_3.jpg" width="200" />
  <img src="https://upload.wikimedia.org/wikipedia/commons/9/9f/Iris_virginica.jpg" width="200" />
</div>

*Images from Wikipedia.*

[1] Fisher, R.A. “The use of multiple measurements in taxonomic problems” Annual Eugenics, 7, Part II, 179-188 (1936).

We can load the data set automatically using the [sklearn](https://scikit-learn.org/stable/) library.

In [ ]:
from sklearn.datasets import load_iris
import pandas as pd

iris_data = load_iris()
df = pd.DataFrame(iris_data.data, columns=iris_data.feature_names)
df['target'] = pd.Series(iris_data.target)

df


## Exercise 1: Review exercises from last week

Like we did last week, please pair off with someone. Record your responses below. Before we reconvene to discuss, in this exercise you will:


1.   Discuss and revise the exercises from last week. Specifically, you should talk through how you solved the problem. The other person will offer suggestions or ask questions if there are parts that are unclear. Revising the exercises should NOT involve directly copying.
2.  Share metrics: what accuracy, precision, recall were you able to achieve? If you tried different hyperparameters or approaches, which model would you choose and why?


## Naive Bayes Classifier
### Revisiting the formulation of Linear Regression

We will start with a *very* familiar idea: **Linear Regression**.

$$
Y = \theta^T X + \epsilon
$$

Here,  
- $Y$ is the output we want to predict (the dependent variable),  
- $X$ is the vector of input features (the independent variables),  
- $\theta$ represents the model parameters (weights), and  
- $\epsilon$ is an error term, which we assume follows a Gaussian distribution.

We usually estimate $\theta$ using **Ordinary Least Squares (OLS)**, by minimizing the squared errors between predictions and true outputs, or equivalently through **Maximum Likelihood Estimation (MLE)** assuming Gaussian noise.

This gives us a clean, deterministic model.  

But notice something: the model only gives a *single prediction*. It doesn’t tell us how confident it is in that prediction. However, we want to use these models in practice! As a result, wouldn’t it be useful if the model could tell us *how uncertain* it is?

*Q: practically, when might we care about uncertainty estimates?*

---

### Probabilistic Reformulation

Instead of treating $Y$ as a single deterministic output, we can take a **probabilistic view**:

$$
Y \sim \mathcal{N}(\theta^T X, \sigma^2),
$$

meaning that for a given input $X$, our model predicts that $Y$ follows a Normal distribution with  
mean $\theta^\top X$ and variance $\sigma^2$. The model therefore predicts both the **expected value** and the **uncertainty** of the output.

In the **Bayesian** framework, we treat the parameters $\theta$ as random variables and update our beliefs about them after seeing data. So we are updating our uncertainty about the model itself, not about the class labels. We represent this using the **posterior** distribution:

$$
p(\theta \mid X, y).
$$

By Bayes’ rule:

$$
p(\theta \mid X, y) = \frac{p(y \mid X, \theta) \, p(\theta)}{p(y \mid X)}
$$

where  
- $p(\theta)$ is the **prior**: what we believe about the parameters before seeing data,  
- $p(y \mid X)$ is the probability of a label given the data,
- $p(y \mid X, \theta)$ is the **likelihood**: how well the parameters explain the observed data, and  
- $p(\theta | X, y)$ is the **posterior**: our updated belief after seeing the data.

This approach provides two main benefits:

* **Priors:** We can include any prior knowledge we have about the model (for example, that $\sigma$ should be small).  
* **Uncertainty:** Instead of producing a single estimate of $\theta$, we get a full posterior distribution that shows how confident we are in different parameter values.

---

### Connecting to Naive Bayes

Now let’s move from regression to **classification**, where the goal is to predict a discrete label $y$ (for example, spam vs. not spam).

In classification, we are interested in finding the probability of a class given the input features:

$$
p(y \mid X).
$$

Using Bayes’ rule, we can express this as:

$$
p(y \mid X) = \frac{p(X \mid y) \, p(y)}{p(X)}.
$$

Here,  
- $p(y \mid X)$ is the **posterior** probability of each class,
- $p(y)$ is the **prior** probability of each class,  
- $p(X | y)$ is the **likelihood** of observing features $X$ given the class `y`, and  
- $p(X)$ is the **evidence** or normalization constant (same for all classes).

Since $p(X)$ is the same for all classes (it doesn’t depend on $y`$, it doesn’t affect which class has the highest probability. So, for prediction, we can ignore it and just find the class that maximizes the numerator:

$$
\hat{y} = \arg\max_y \, p(X \mid y) \, p(y).
$$


The **Naive Bayes assumption** is that all features are conditionally independent given the class label. We make this independence assumption to make the model simple and practical. Without it, we would need to estimate the full joint probability $p(X | y)$ (which grows exponentially with the number of features). That would require an unrealistic amount of data!

So we assume:
$$
p(X \mid y) = \prod_{i=1}^d p(x_i \mid y).
$$



## Exercise 2: Flower classification with Naive Bayes

Let’s revisit the Iris flower classification task and build a **Naive Bayes classifier** that predicts which Iris species a sample belongs to (setosa, versicolor, or virginica). In this model, each class will have its own probability distribution over the features. We’ll represent that distribution using a Gaussian (Normal) distribution for each feature.

As the first step, you will:
   - Load the Iris dataset (e.g., using `sklearn.datasets.load_iris()`).
   - Split the data into features $X$ and labels $y$.
   - Separate the training data into three subsets: one for each class label (0, 1, and 2).  
     Each subset should contain only the samples belonging to that class.


In [ ]:
## To-do: Extract indices from the training set that have samples from class 0, 1, 2

## To-do: Separate train_features into train_features_classX for each class using the indices

Next, you will take the separate collection of examples representing each individual flower class and their formal features to estimate the statistical properties of each random variable representing the flower type. We will use Normal distributions to represent each unique flower distribution.

In [ ]:
## To-do: Write a function for computing the mean of a random variable
# Assume x is the input with multiple samples and features
# Note that each mean corresponding to the average of each individual flower feature should be computed individually
def expval(x):
  return -1


## To-do: Write a function for computing the scale for a random variable
# Assume x is the input with multiple samples and mean is the mean of x
# Note that each std. dev. computation corresponding to the average of each individual flower feature should be computed individually
# If any var is 0 add a small constant (e.g., 1e-6) to prevent division errors later.
def var(x, mu):
  return -1


Now, using the functions you implemented, summarize the statistical properties of each class distribution with their mean and standard deviation. You should plot the Normal distributions representing each class feature values distribution. This helps visualize how well your Gaussian fits the data by evaluating your samples under the distribution.

In [ ]:
from torch.distributions.normal import Normal
import matplotlib
import matplotlib.pyplot as plt

def gaussian(mean, st_dev):
  return Normal(mean, st_dev)

# To-do: Plot the distributions

Now, calculate the prior probability $p(y=k)$ for each class as the fraction of training samples belonging to that class.


In [ ]:
# To-do: Calculate prior

Next, for a new sample, compute the likelihood $p(X \mid y=k)$ for each class using the Gaussian probability formula. Multiply each likelihood by its corresponding prior $p(y=k)$. Select the class that gives the highest resulting value: this is your predicted label.


In [ ]:
# To-do: Prediction function

You now have your classifier! Test your classifier on several samples.
Compare your predicted labels with the true labels to see how well it performs.

* Compare your predicted labels with the true labels to see how well it performs.
*   Evaluate the accuracy, precision, and recall of the model.



In [ ]:
# To-do: Write functions to compare and plot